In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
%cd /content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1

/content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1


In [ ]:
!pip install transformers evaluate datasets
!pip install accelerate sentencepiece
!pip install lm-evaluation-harness


In [ ]:
!pip install "lm-evaluation-harness[math,ifeval,sentencepiece]"
!pip install transformers accelerate datasets evaluate sentencepiece sacrebleu


In [50]:
!mkdir -p /content/custom_tasks/squad_v1
!cp /content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1/custom_tasks/squad_v1/* /content/custom_tasks/squad_v1/

cp: -r not specified; omitting directory '/content/drive/MyDrive/TUM/Pratikum/lm-evaluation-harnessSquadV1.1/custom_tasks/squad_v1/__pycache__'


In [14]:
from lm_eval.tasks import TaskManager
tm = TaskManager(include_path="/content/custom_tasks",
                 include_defaults = False)
print(list(tm.all_tasks)[:20])  # should list squadv1


['squadv1']


In [ ]:
!pip install -e ".[dev]"

In [49]:
!rm -r /content/custom_tasks/

In [ ]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [15]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
import lm_eval
from lm_eval.tasks import TaskManager

# if your squadv1.yaml is inside the installed lm_eval/tasks tree,
# you don't need include_path
tm = TaskManager()

print("squadv1" in tm.all_tasks)      # or whatever name you put in `task:` in the yaml

True


In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import TaskManager

tm = TaskManager(
    include_path="/content/drive/MyDrive/TUM/Praktikum/lm-evaluation-harnessSquadV1.1/lm_eval/tasks/squad_v1"
)

model = HFLM(pretrained="facebook/opt-125m")

results = lm_eval.simple_evaluate(
    model,
    tasks=["squadv1"],   # this is the `task:` name in your YAML
    num_fewshot=0,
    batch_size=64,
    task_manager=tm,     # optional if your task is inside the installed lm_eval tree
)

print(results["results"])


INFO - NumExpr defaulting to 12 threads.
INFO - TensorFlow version 2.19.0 available.
INFO - JAX version 0.7.2 available.


In [ ]:
import squad_v1.task
importlib.reload(squad_v1.task)

In [25]:
!pip install sqlitedict

  Preparing metadata (setup.py) ... done
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=07ee593796180c66bdd1b8d6a6f60a6e9cb1ffa9fc30e7301a19db58c208816f
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21/fc016aef45ffcabe27129a2252f061387cbf278d2086225a64
Successfully built sqlitedict


In [53]:
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-1.3b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 32
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos.\nThe Denver Broncos defeated the Carolina Panthers 24–10 in Super Bowl 50.\nThe Denver Broncos defeated the Carolina Panthers 24–10 in'



  5%|▌         | 5/100 [00:02<00:49,  1.92it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th Super Bowl logo was gold.\n\nReferences\n\nExternal links\n\nSuper Bowl 50'



100%|██████████| 100/100 [00:51<00:00,  1.94it/s]


Scores: {'exact_match': 0.0, 'f1': 13.358174968017229}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual.json


In [54]:
#we put a max_tokens smaller in order to have more exact match
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-1.3b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 8
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda
Evaluating on 100 examples


  1%|          | 1/100 [00:00<00:15,  6.20it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the AFC at Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos.\n\nThe Denver'


  2%|▏         | 2/100 [00:00<00:15,  6.32it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the NFC at Super Bowl 50?
GT  : ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
PRED: 'The Carolina Panthers.\n\nThe Carolina'


...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The Denver Broncos.\nThe Denver Broncos'


  5%|▌         | 5/100 [00:00<00:14,  6.48it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th Super Bowl'


100%|██████████| 100/100 [00:14<00:00,  6.82it/s]


Scores: {'exact_match': 2.0, 'f1': 36.4576479076479}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual.json


In [56]:
#different prompt to get smaller answers
#we put a max_tokens smaller in order to have more exact match
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "facebook/opt-1.3b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 8
LIMIT = 100  # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_s0.00_squadv1_manual.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# OPT often has no pad token, so use eos as pad
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer with a short phrase (no explanation):"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = out[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text,
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda
Evaluating on 100 examples


  1%|          | 1/100 [00:00<00:16,  6.17it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the AFC at Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The answer is:'


  2%|▏         | 2/100 [00:00<00:15,  6.25it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the NFC at Super Bowl 50?
GT  : ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
PRED: 'The answer is:'


...

==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'The answer is:'


  5%|▌         | 5/100 [00:00<00:14,  6.59it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'The color of the 50th'


100%|██████████| 100/100 [00:14<00:00,  6.82it/s]


Scores: {'exact_match': 0.0, 'f1': 18.63181818181818}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_s0.00_squadv1_manual.json
